In [1]:
## Restructured model with germany data

In [2]:
import pickle
with open('../datasets/data/polymod_germany.pkl', 'rb') as file:
    data = pickle.load(file)

In [3]:
from cntmosaic.dataloader import PandasLoader

In [4]:
import pandas as pd
import numpy as np

In [5]:
import xarray as xr

In [6]:
from cntmosaic.dataloader import CoordToColumns

In [7]:
mapp = CoordToColumns(age_part='age_part', id_var='id', age_cnt='age_cnt', y='y')

In [8]:
d = PandasLoader(data['contacts'], mapp)

In [9]:
d.load()

In [10]:
d.ds

<xarray.Dataset> Size: 74MB
Dimensions:   (id: 1281, age_part: 85, age_cnt: 85)
Coordinates:
  * id        (id) int64 10kB 846 847 848 849 850 ... 2180 2181 2182 2184 2185
  * age_part  (age_part) int64 680B 0 1 2 3 4 5 6 7 ... 77 78 79 80 81 82 83 84
  * age_cnt   (age_cnt) int64 680B 0 1 2 3 4 5 6 7 8 ... 77 78 79 80 81 82 83 84
Data variables:
    y         (id, age_part, age_cnt) int64 74MB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0

In [12]:
d.raw_df

,y,id,age_part,age_cnt
0,3.0,846,10.0,43.0
1,2.0,846,10.0,70.0
2,2.0,846,10.0,68.0
3,1.0,846,10.0,11.0
4,2.0,846,10.0,13.0
...,...,...,...,...
10648,1.0,2185,70.0,57.0
10649,1.0,2185,70.0,57.0
10650,1.0,2185,70.0,61.0
10651,1.0,2185,70.0,32.0


In [11]:
d.ds.sel(id=846, age_part=10, age_cnt=43)

Format,coo
Data Type,float64
Shape,()
nnz,1
Density,1.0
Read-only,True
Size,8
Storage ratio,1.00


In [13]:
d.ds.groupby(['age_part', 'age_cnt'])

<DatasetGroupBy, grouped over 2 grouper(s), 7225 groups in total:
    'age_part': 85 groups with labels 0, 1, 2, 3, 4, 5, ..., 80, 81, 82, 83, 84
    'age_cnt': 85 groups with labels 0, 1, 2, 3, 4, 5, ..., 80, 81, 82, 83, 84>

In [ ]:
from cntmosaic.models.restr_BRCfine import restr_BRCfine

In [ ]:
model = restr_BRCfine(out)

In [ ]:
model.compile()

In [ ]:
import jax
model.run_inference_mcmc(jax.random.PRNGKey(0),
    num_samples = 50,
    num_warmup = 50,
    num_chains = 2)

In [ ]:
from cntmosaic.sim import ModelEvaluatorMCMC

In [ ]:
e = ModelEvaluatorMCMC(model)

In [ ]:
e.post = model.mcmc.get_samples(group_by_chain=True)
#e.get_posterior() extra dimension for different chains

In [ ]:
import numpy as np

In [ ]:
log_rate = e.post['log_rate']
post_cint = {}
for name, site in e.post.items():
    if 'log_delta' in name:
        var = name.split('/')[0]
        cat = e.model.data[var].cat.categories
        post_cint[var] = {
            cat[i]: np.exp(log_rate[:, None, :, :] + site + e.model._precompute.log_P[None, None, :, :])[:, i, :, :]
            for i in range(len(cat))
        }

In [ ]:
e.post.keys()

In [ ]:
e.post['inv_disp'].shape

In [ ]:
import numpyro
import jax.numpy as jnp
from numpyro import distributions as dist

def guide():
    # --- beta0: scalar Normal(0, 10) ---
    beta0_loc = numpyro.param("beta0_loc", jnp.zeros(()))
    beta0_scale = numpyro.param("beta0_scale", jnp.ones(()), constraint=dist.constraints.positive)
    numpyro.sample("beta0", dist.Normal(beta0_loc, beta0_scale))

    # --- alpha: scalar InverseGamma(5, 5) ---
    alpha_conc = numpyro.param("alpha_conc", jnp.ones(()), constraint=dist.constraints.positive)
    alpha_rate = numpyro.param("alpha_rate", jnp.ones(()), constraint=dist.constraints.positive)
    numpyro.sample("baseline", dist.InverseGamma(alpha_conc, alpha_rate))

    # --- rho: vector of 2 InverseGamma(5, 5) ---
    rho_conc = numpyro.param("rho_conc", jnp.ones(2), constraint=dist.constraints.positive)
    rho_rate = numpyro.param("rho_rate", jnp.ones(2), constraint=dist.constraints.positive)
    numpyro.sample("rho", dist.InverseGamma(rho_conc, rho_rate))

    inv_disp_rate = numpyro.param("inv_disp_rate", jnp.ones(len(out)), constraint=dist.constraints.positive)
    numpyro.sample("inv_disp", dist.Exponential(inv_disp_rate))


In [ ]:
model.run_inference_svi(jax.random.PRNGKey(0),guide, num_steps = 1000)

In [ ]:
from cntmosaic.sim import ModelEvaluatorSVI
svi = ModelEvaluatorSVI(model)

In [ ]:
svi.get_pred_cint()

In [ ]:
svi.pred_cint